# Test prepare evaluation on Colab to leverage CUDA GPU


## Setup Repo

- The project has previously been imported via github, and the data folder with additional files uploaded manually
- The project is located under `drive/MyDive/project/ms-project`


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd drive/MyDrive/project/ms-project

## Install dependencies

The original repo is using uv, but it is not suited to run on Colab with the Cuda devices so we install dependencies with pip instead


In [ ]:
!pip install torch
!pip install numpy
!pip install pillow
!pip install tqdm
!pip install colpali-engine
!pip install python-dotenv
!pip install pdf2image
!pip install datasets
!pip install mteb
!pip install ir-measures
!pip install einops
!pip install transformers
!pip install ollama
!pip install aiohttp
!pip install psutil
!pip install colab-xterm


## Run Ollama for the generation model

Ollama needs to run for the generation model qwen2.5vl:7b to be used, the following commands need to be entered in the terminal created below.

- `curl https://ollama.ai/install.sh | sh`
- `ollama serve &`
- `ollama pull qwen2.5vl:7b`


In [ ]:
%load_ext colabxterm
%xterm

## Reload imports

If a py file is edited, it won't import the revised version but the cache one.

This script enforces an import reload


In [ ]:
import importlib
import sys


def recursive_reload(package_name):
    """Recursively reload all modules in a given package. Useful for Colab or Jupyter after editing .py files."""
    modules_to_reload = [name for name in sys.modules if name.startswith(package_name)]

    for module_name in sorted(modules_to_reload, key=len, reverse=True):
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded: {module_name}")

## Load the test script


In [ ]:
%cd src

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add src path for imports
src_path = Path.cwd()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

load_dotenv()

# Load and reload with recursive call
from pipeline.rags.factory_rag import RAGFactory

recursive_reload("pipeline")
from pipeline.rags.factory_rag import RAGFactory


def verify_dataset(dataset_path: Path):
    if not dataset_path.exists():
        msg = f"Dataset {dataset_path} not found."
        raise FileNotFoundError(msg)


def verify_configs(config_path: Path):
    if not config_path.exists():
        msg = f"Config {config_path} not found."
        raise FileNotFoundError(msg)


async def main(dataset_name: str, rag_configs: str):
    """
    Create a new RAG system and index an evaluation dataset.

    Usage:
    uv run evaluation/prepare.py --dataset-name "sherpa/consulting_light_dataset" --rag-configs "multimodal_colqwen"
    """

    # Params validation
    dataset_path = src_path / "data/evaluation/datasets" / dataset_name
    dataset_path.mkdir(parents=True, exist_ok=True)
    verify_dataset(dataset_path)

    rag_configs_path = src_path / "configs" / f"{rag_configs}.json"
    verify_configs(rag_configs_path)

    # Create RAG system
    # Configuration
    evaluation_dir = os.getenv("EVALS_DATA_DIR")
    if evaluation_dir is None:
        # Fallback to default if environment variable is not set
        evaluation_dir = str(src_path / "data/evaluation/datasets")
        print(f"EVALS_DATA_DIR not set, using default: {evaluation_dir}")

    evaluation_dir = Path(evaluation_dir)

    data_dir = os.getenv("RAGS_DATA_DIR")
    if data_dir is None:
        # Fallback to default if environment variable is not set
        data_dir = str(src_path / "data/rags")
        print(f"RAGS_DATA_DIR not set, using default: {data_dir}")

    data_dir = Path(data_dir)

    # Load RAG configs
    rag_configs = {}
    with rag_configs_path.open("r") as f:
        rag_configs = json.load(f)

    # Initialize the RAG using the factory
    evaluation_rag = RAGFactory.create_rag(rag_configs, data_dir)

    # Extract metadata from corpuses
    documents = list(
        (evaluation_dir / rag_configs["configs"]["knowledge_base"] / "corpuses").glob(
            "*.jpg",
        ),
    )

    print("Starting extraction...")
    await evaluation_rag.extract(documents, preprocessed=True, batch_size=8)
    print("Extraction completed!")

    # Index all corpuses
    print("Starting indexing...")
    await evaluation_rag.index()
    print("Indexing completed!")

    print(f"Dataset ready for evaluation in '{rag_configs['name']}'.")


## Run


In [ ]:
await main(
    dataset_name="vidore/arxivqa_test_subsampled_beir", rag_configs="multimodal_page"
)